# Blues Lick Finder: RAG Baseline

A minimal RAG baseline over blues guitar lesson videos, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Given a song or technique, it retrieves candidate lessons and asks the LLM to rank them by teaching quality.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index
from gitsource import chunk_documents

openai_client = OpenAI()

## 1. Load Your Data

Each document is one YouTube lesson video: metadata from a `yt-dlp` search (no API key needed) plus the English transcript from `youtube-transcript-api` when one exists. Each video also gets its top 20 viewer comments by likes, excluding the creator's own.

The first run fetches from YouTube and caches everything to `data/lessons.json`. Later runs load from that file. Delete it to re-fetch.

In [2]:
from pathlib import Path

import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi

DATA_PATH = Path("../data/lessons.json")

TOPICS = [
    "BB King box blues lick lesson",
    "Freddie King Hideaway guitar lesson",
    "Stevie Ray Vaughan Pride and Joy guitar lesson",
    "blues turnaround licks lesson",
    "blues string bending technique lesson",
    "minor pentatonic blues licks beginner lesson",
    "Texas blues shuffle rhythm guitar lesson",
    "slow blues soloing lesson",
]
RESULTS_PER_TOPIC = 5

transcript_api = YouTubeTranscriptApi()


def fetch_transcript(video_id):
    try:
        transcript = transcript_api.fetch(video_id, languages=["en"])
    except Exception:
        return None
    return " ".join(s.text for s in transcript.snippets)


def fetch_lessons(topics, results_per_topic):
    lessons = {}
    ydl_opts = {"quiet": True, "no_warnings": True, "skip_download": True}

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for topic in topics:
            print(f"Searching: {topic}")
            result = ydl.extract_info(f"ytsearch{results_per_topic}:{topic}", download=False)

            for entry in result["entries"]:
                video_id = entry["id"]
                if video_id in lessons:
                    continue

                lessons[video_id] = {
                    "video_id": video_id,
                    "url": f"https://www.youtube.com/watch?v={video_id}",
                    "title": entry.get("title"),
                    "channel": entry.get("channel"),
                    "description": (entry.get("description") or "")[:1000],
                    "tags": " ".join(entry.get("tags") or []),
                    "duration_sec": entry.get("duration"),
                    "view_count": entry.get("view_count"),
                    "topic": topic,
                    "transcript": fetch_transcript(video_id),
                }

    return list(lessons.values())


COMMENTS_PER_VIDEO = 20


def fetch_comments(video_ids):
    # Top comments by likes, skipping replies and the creator's own (usually promo) comments.
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "getcomments": True,
        "extractor_args": {"youtube": {"max_comments": ["50", "50", "0", "0"], "comment_sort": ["top"]}},
    }

    comments_by_video = {}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for video_id in video_ids:
            info = ydl.extract_info(f"https://www.youtube.com/watch?v={video_id}", download=False)
            comments = [
                {"text": c["text"], "likes": c.get("like_count") or 0}
                for c in info.get("comments") or []
                if c.get("parent") == "root" and not c.get("author_is_uploader")
            ]
            comments_by_video[video_id] = comments[:COMMENTS_PER_VIDEO]

    return comments_by_video


if DATA_PATH.exists():
    documents = json.loads(DATA_PATH.read_text())
else:
    documents = fetch_lessons(TOPICS, RESULTS_PER_TOPIC)
    DATA_PATH.write_text(json.dumps(documents, indent=2))

missing = [doc["video_id"] for doc in documents if "comments" not in doc]
if missing:
    print(f"Fetching comments for {len(missing)} videos")
    comments_by_video = fetch_comments(missing)
    for doc in documents:
        if doc["video_id"] in comments_by_video:
            doc["comments"] = comments_by_video[doc["video_id"]]
    DATA_PATH.write_text(json.dumps(documents, indent=2))

with_transcript = sum(1 for d in documents if d["transcript"])
with_comments = sum(1 for d in documents if d["comments"])
print(f"Loaded {len(documents)} lessons, {with_transcript} with transcripts, {with_comments} with comments")

Loaded 40 lessons, 27 with transcripts, 39 with comments


## 2. Build the Index

Transcripts are long, so we split them into overlapping chunks with `chunk_documents` (3000 characters, step 1500), the same way we did in class, and index the chunks instead of whole videos. Each chunk keeps its video's metadata. Videos without a transcript are skipped.

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
transcript_docs = [doc for doc in documents if doc["transcript"]]
document_chunks = chunk_documents(
    transcript_docs, size=3000, step=1500, content_field_name="transcript"
)
print(f"{len(transcript_docs)} videos -> {len(document_chunks)} chunks")

index = Index(
    text_fields=["title", "description", "tags", "content"],
    keyword_fields=["video_id", "channel", "topic"],
)

index.fit(document_chunks)

27 videos -> 142 chunks


## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    results = index.search(query, num_results=50)

    # Results are sorted best-first and may hold several chunks of the same
    # video. Keep only the first (best) chunk per video so we return 5
    # different videos.
    best_chunk_per_video = {}
    for chunk in results:
        best_chunk_per_video.setdefault(chunk["video_id"], chunk)

    return list(best_chunk_per_video.values())[:5]

In [5]:
instructions = """
You're a blues guitar teacher helping a student find the best lesson videos
for the song or technique in their QUERY. YouTube ranks videos by engagement;
your job is to rank them by teaching quality instead.

Use only the candidate lessons in the CONTEXT. Skip candidates that aren't
about the requested song or technique. Rank the rest from best to worst
using this rubric, judged mostly from each candidate's `content` field,
which is the part of the video's transcript that best matches the QUERY:

- Breakdown: splits the lick into small, clearly explained pieces
- Demonstration: plays it slowly before playing it at full speed
- Specifics: names frets, strings, fingers, or points to tab/notation
- Why: explains the theory or feel (scale, chord tones, phrasing) behind it
- Focus: the excerpt teaches the lick instead of intros or sales pitches

Each candidate also has a `comments` field with its most-liked viewer
comments and like counts. Use comments that talk about the teaching (pace,
clarity, tabs, how quickly it gets going, mistakes) as extra evidence for
the rubric. Ignore general praise, jokes, and song requests. An empty list
means no comments are available.

When you use a comment, quote it word for word with its like count. Never
describe or summarize what viewers said without quoting them, and never
claim something about the comments that no comment actually says.

Also estimate the lesson's level from the title, tags, transcript excerpt,
and comments:

- beginner: explains basics like what a bend or the pentatonic box is,
  plays very slowly, or says it's for people starting out
- intermediate: assumes the pentatonic box and basic bends are known and
  focuses on phrasing, licks in context, and following the chord changes
- advanced: chord-tone soloing, fast or complex runs, little explanation
  of the basics

Give one short reason for the level, quoting the transcript or a comment
when you can. If there isn't enough to tell, say "unclear" instead of
guessing.

For each ranked lesson, return: rank, title, channel, URL, a score from 1 to 5,
a one-sentence justification that cites the rubric, viewer evidence (1-2
quoted comments with like counts, or "none" if no comment is about the
teaching), level with a short reason, and practice-log tags (song,
technique, key if mentioned). If none of the candidates fit the QUERY,
say so.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUERY>
{query}
</QUERY>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("Which lessons teach the BB King box licks best?"))

Here are the ranked lessons teaching BB King box licks based on teaching quality:

### 1. 
**Title:** Learn 10 Classic Blues Licks in the BB King Box!  
**Channel:** swiftlessons  
**URL:** [Watch here](https://www.youtube.com/watch?v=i6Ho6aubraw)  
**Score:** 5  
**Justification:** This lesson thoroughly breaks down multiple licks in the BB King box, with precise details on frets and strings while demonstrating slowly first.  
**Viewer Evidence:** "one of the best teachers on the net, very usable and slow enough to learn, not just a fast short tease" (16 likes).  
**Level:** Intermediate - "I'm going to teach you 10 licks," indicating some prior knowledge is assumed.  
**Tags:** technique, song

---

### 2. 
**Title:** The BB King Box & Beyond - Blues Lead Guitar Lesson  
**Channel:** swiftlessons  
**URL:** [Watch here](https://www.youtube.com/watch?v=pZz-QHt8UBc)  
**Score:** 4  
**Justification:** This lesson covers the BB King box's position and adjacent scales, providing clarity 